# Hito 4 · Entrenamiento en un notebook de SageMaker y almacenamiento en S3

Datos de venta (Datos.csv). No crea un endpoint ni un job gestionado de entrenamiento.

> **Versión para portfolio.** Trabajo académico de David Díaz Vílchez. Las salidas conservadas proceden de la entrega original; no se han reejecutado los entrenamientos al preparar el repositorio. Se han retirado muestras de anuncios, metadatos de sesión y salidas de infraestructura. Consulta `../docs/resultados.md` y `../docs/reproduccion.md`.

> En este cuaderno se ha corregido la imputación para ajustarla solo sobre entrenamiento y guardar el pipeline completo. Sus salidas se han limpiado: las métricas históricas de la documentación pertenecen a la versión anterior, no a esta corrección.


In [ ]:
# Autenticación mediante el rol del notebook o la cadena estándar de AWS.
# No pegues credenciales aquí. No se enumeran buckets de la cuenta.
import os
import boto3
BUCKET = os.environ.get("PROJECT_S3_BUCKET")
KEY = os.environ.get("PROJECT_S3_DATA_KEY", "raw/Datos.csv")
MODEL_KEY = os.environ.get("PROJECT_S3_MODEL_KEY", "model/linear_pipeline.joblib")
if not BUCKET:
    raise ValueError("Define PROJECT_S3_BUCKET con tu propio bucket antes de continuar.")
s3 = boto3.client("s3")

In [ ]:
import pandas as pd

df = pd.read_csv(f"s3://{BUCKET}/{KEY}")
print("Dimensiones:", df.shape)

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

In [ ]:
# seleccionamos columnas útiles
cols_utiles = [
    "PrecioActual",
    "PrecioAnterior",
    "metros",
    "habitaciones",
    "baños"
]

df_model = df[cols_utiles].copy()
df_model.head()

### Tratamos valores nulos

In [ ]:
df_model.isna().sum()

In [ ]:
# La imputación se ajusta después de dividir los datos, dentro del pipeline.
# Así se evita usar información del test al calcular la media.

In [ ]:
df_model.isna().sum()

In [ ]:
# Separamos variables X e y
X = df_model.drop("PrecioActual", axis=1)
y = df_model["PrecioActual"]

X.head(), y.head()

### Entrenar el modelo de Machine Learning (Scikit-Learn)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("regressor", LinearRegression()),
])
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

mae, rmse, r2

### Guardar el pipeline (preprocesamiento + modelo)

In [ ]:
# Guardar modelo localmente
import joblib
import os

os.makedirs("model", exist_ok=True)
joblib.dump(model, "model/linear_pipeline.joblib")

In [ ]:
# Escritura opcional: requiere autorización de escritura sobre esa clave S3.
if os.environ.get("PROJECT_UPLOAD_MODEL") == "1":
    s3.upload_file("model/linear_pipeline.joblib", BUCKET, MODEL_KEY)
    print("Pipeline guardado en S3.")
else:
    print("Subida desactivada. Define PROJECT_UPLOAD_MODEL=1 para activarla.")

In [ ]:
if os.environ.get("PROJECT_UPLOAD_MODEL") == "1":
    metadata = s3.head_object(Bucket=BUCKET, Key=MODEL_KEY)
    print("Tamaño del artefacto:", metadata["ContentLength"], "bytes")